In [ ]:
! pip -q install -U ipywidgets faiss-cpu sentence-transformers rank_bm25 scikit-learn openai anthropic google-genai pymupdf nltk transformers accelerate torch

In [ ]:
# Import packages
import os
import re
import json
import time
import fitz
import faiss
import torch
import numpy as np
import ipywidgets as widgets

from pathlib import Path
from collections import defaultdict, deque, Counter
from typing import Dict, Any, List, Tuple
from IPython.display import display, HTML, clear_output

from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

from sentence_transformers import SentenceTransformer, CrossEncoder
from openai import OpenAI
from anthropic import Anthropic
from google import genai
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
# Basic settings

# File paths for three directionaries
PDF_DIR = ""
DIVISIONTREE_DIR = ""
CHUNKS_DIR = ""


OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")


# Model selection
MODEL_REGISTRY = {
    "GPT-5": {
        "provider": "openai",
        "model_name": "gpt-5"
    },
    "Claude Opus 4": {
        "provider": "anthropic",
        "model_name": "claude-opus-4-1"
    },
    "Gemini-3": {
        "provider": "google",
        "model_name": "gemini-2.5-pro"
    },
    "Qwen-3": {
        "provider": "qwen_local",
        "model_name": "Qwen/Qwen3-4B-Instruct-2507"
    }
}

DEFAULT_MODEL_LABEL = "GPT-5"

# Parameters for division identification
TOP_K_DIV = 5
RRF_K = 60
USE_WEIGHTED_FUSION = False
# WEIGHTS = {"sem": 0.6, "bm25": 0.25, "kw": 0.15}

#Parameter for transversal retrieval
SEED_TOP_N = 8
D_MAX = 2
N_MAX = 120
FINAL_TOP_K = 5

# Model loading
openai_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None
google_client = genai.Client(api_key=GOOGLE_API_KEY) if GOOGLE_API_KEY else None

QWEN_TOKENIZER = None
QWEN_MODEL = None

def load_qwen_once(model_name: str):
    global QWEN_TOKENIZER, QWEN_MODEL
    if QWEN_TOKENIZER is None or QWEN_MODEL is None:
        print(f"[Info] Loading local Qwen model: {model_name}")
        QWEN_TOKENIZER = AutoTokenizer.from_pretrained(model_name)
        QWEN_MODEL = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype="auto",
            device_map="auto"
        )
    return QWEN_TOKENIZER, QWEN_MODEL

In [ ]:
# Directionary processing

def load_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def extract_division_number(name: str):
    """
    Extract division number from names like:
    - Division 10.0 Steel work.pdf
    - doctree_all_contents_division_10.json
    - division_10_all_chunks_text_table_figure.json
    """
    s = Path(name).stem.lower()
    patterns = [
        r"division[\s_]+(\d+)(?:\.\d+)?",
        r"division_(\d+)",
        r"division[\s]+(\d+)",
    ]
    for pat in patterns:
        m = re.search(pat, s)
        if m:
            return str(int(m.group(1)))
    return None

def load_pdf_text(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    parts = []
    for _, page in enumerate(doc):
        txt = page.get_text("text")
        if txt:
            parts.append(txt)
    doc.close()
    return "\n".join(parts)

def scan_division_assets(pdf_dir: str, doctree_dir: str, chunks_dir: str):
    pdf_files = sorted(Path(pdf_dir).glob("*.pdf"))
    doctree_files = sorted(Path(doctree_dir).glob("*.json"))
    chunk_files = sorted(Path(chunks_dir).glob("*.json"))

    pdf_map = {}
    for p in pdf_files:
        div_no = extract_division_number(p.name)
        if div_no is not None:
            pdf_map[div_no] = str(p)

    doctree_map = {}
    for p in doctree_files:
        div_no = extract_division_number(p.name)
        if div_no is not None:
            doctree_map[div_no] = str(p)

    chunk_map = {}
    for p in chunk_files:
        div_no = extract_division_number(p.name)
        if div_no is not None:
            chunk_map[div_no] = str(p)

    common_divs = sorted(
        set(pdf_map.keys()) & set(doctree_map.keys()) & set(chunk_map.keys()),
        key=lambda x: int(x)
    )

    records = []
    for div_no in common_divs:
        pdf_path = pdf_map[div_no]
        doctree_path = doctree_map[div_no]
        chunks_path = chunk_map[div_no]

        try:
            pdf_text = load_pdf_text(pdf_path)
            doctree = load_json(doctree_path)
            chunks = load_json(chunks_path)
            if isinstance(chunks, dict) and "chunks" in chunks:
                chunks = chunks["chunks"]

            records.append({
                "division_no": div_no,
                "pdf_path": pdf_path,
                "pdf_name": Path(pdf_path).name,
                "document_title": doctree.get("document_title", Path(pdf_path).stem),
                "doctree_path": doctree_path,
                "chunks_path": chunks_path,
                "pdf_text": pdf_text,
                "doctree": doctree,
                "chunks": chunks
            })
        except Exception as e:
            print(f"[Warning] Failed loading division {div_no}: {e}")

    if not records:
        raise ValueError("No aligned PDF/doctree/chunk triples found. Check naming format.")

    return records

division_records = scan_division_assets(PDF_DIR, DOCTREE_DIR, CHUNKS_DIR)
print(f"Loaded {len(division_records)} aligned divisions.")
for r in division_records:
    print(f"Division {r['division_no']}: {r['pdf_name']}")

# Read from division tree directionary

def iter_nodes_plain(doctree: dict):
    stack = list(doctree.get("sections", []) or [])
    while stack:
        node = stack.pop()
        if not isinstance(node, dict):
            continue
        yield node
        subs = node.get("subsections", []) or []
        for child in reversed(subs):
            stack.append(child)

def node_text(node: Dict[str, Any]) -> str:
    blocks = node.get("text_blocks", []) or []
    return "\n".join([b.strip() for b in blocks if isinstance(b, str) and b.strip()])

def iter_nodes_with_internal_keys(doctree: Dict[str, Any]):
    sections = doctree.get("sections", []) or []
    stack = []

    for idx, sec in enumerate(reversed(sections)):
        real_idx = len(sections) - 1 - idx
        stack.append((sec, None, f"sec_{real_idx}"))

    while stack:
        node, parent_key, internal_key = stack.pop()

        if not isinstance(node, dict):
            continue

        yield {
            "internal_key": internal_key,
            "raw_id": node.get("id", ""),
            "title": node.get("title", ""),
            "parent_internal_key": parent_key,
            "node": node
        }

        subs = node.get("subsections", []) or []
        for i in range(len(subs) - 1, -1, -1):
            child = subs[i]
            child_key = f"{internal_key}/sub_{i}"
            stack.append((child, internal_key, child_key))

def build_tree_maps_safe(doctree: Dict[str, Any]):
    internal_parent_map = {}
    internal_children_map = defaultdict(list)
    internal_to_raw = {}
    raw_to_internal = defaultdict(list)

    for item in iter_nodes_with_internal_keys(doctree):
        ikey = item["internal_key"]
        raw_id = item["raw_id"]
        pkey = item["parent_internal_key"]

        internal_parent_map[ikey] = pkey
        internal_to_raw[ikey] = raw_id
        raw_to_internal[raw_id].append(ikey)

        if pkey is not None:
            internal_children_map[pkey].append(ikey)

    return internal_parent_map, internal_children_map, internal_to_raw, raw_to_internal

def is_structured_regulation_id(x: str) -> bool:
    x = str(x or "").strip()
    return bool(re.search(r"\d", x))

In [ ]:
# Query-guided construction work division identification

def tokenize_for_bm25(text: str):
    return re.findall(r"\w+", (text or "").lower())

def split_text_into_passages(text: str, max_chars: int = 1200):
    paras = re.split(r"\n\s*\n", text or "")
    paras = [p.strip() for p in paras if p.strip()]
    if not paras:
        return [text[:max_chars]] if text else []

    out = []
    cur = []
    cur_len = 0

    for p in paras:
        if cur_len + len(p) > max_chars and cur:
            out.append(" ".join(cur))
            cur = [p]
            cur_len = len(p)
        else:
            cur.append(p)
            cur_len += len(p)

    if cur:
        out.append(" ".join(cur))
    return out

print("[Info] Loading division-level embedding model...")
div_emb_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

division_texts = [r["pdf_text"] for r in division_records]
division_passages = [split_text_into_passages(t) for t in division_texts]

div_passage_embs = [
    div_emb_model.encode(ps, convert_to_numpy=True, normalize_embeddings=True)
    if len(ps) > 0 else np.zeros((0, 384))
    for ps in division_passages
]

bm25_corpus = [tokenize_for_bm25(t) for t in division_texts]
bm25 = BM25Okapi(bm25_corpus)

tfidf = TfidfVectorizer(max_features=7000, ngram_range=(1, 2), stop_words="english")
kw_tfidf_mat = tfidf.fit_transform(division_texts)
kw_tfidf_mat = normalize(kw_tfidf_mat, norm="l2", axis=1)

def rrf_fusion(rank_lists, k=60):
    n = len(division_records)
    scores = np.zeros(n, dtype=float)
    for ranks in rank_lists:
        for r, idx in enumerate(ranks):
            scores[idx] += 1.0 / (k + r + 1)
    return scores

def rank_divisions_improved(query: str, top_n=5, rrf_k=60, use_weighted=False, weights=None):
    q = (query or "").strip()
    if not q:
        return [], None

    # semantic maxsim
    q_emb = div_emb_model.encode([q], convert_to_numpy=True, normalize_embeddings=True)
    sem_scores = np.zeros(len(division_records), dtype=float)
    for i in range(len(division_records)):
        P = div_passage_embs[i]
        sem_scores[i] = float(np.max(P @ q_emb.T)) if P.shape[0] > 0 else 0.0

    # BM25
    bm25_scores = np.array(bm25.get_scores(tokenize_for_bm25(q)), dtype=float)
    if bm25_scores.max() > bm25_scores.min():
        bm25_norm = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min())
    else:
        bm25_norm = bm25_scores * 0.0

    # TF-IDF keyword
    q_kw = tfidf.transform([q])
    q_kw = normalize(q_kw, norm="l2", axis=1)
    kw_scores = cosine_similarity(kw_tfidf_mat, q_kw).reshape(-1)

    # Fusion
    if use_weighted:
        final = weights["sem"] * sem_scores + weights["bm25"] * bm25_norm + weights["kw"] * kw_scores
    else:
        r_sem = np.argsort(-sem_scores)
        r_bm25 = np.argsort(-bm25_scores)
        r_kw = np.argsort(-kw_scores)
        final = rrf_fusion([r_sem, r_bm25, r_kw], k=rrf_k)

    idxs = np.argsort(-final)[:top_n]
    out = []
    for rank, i in enumerate(idxs, start=1):
        rec = division_records[i]
        out.append({
            "rank": rank,
            "division_index": int(i),
            "division_no": rec["division_no"],
            "pdf_name": rec["pdf_name"],
            "document_title": rec["document_title"],
            "doctree_path": rec["doctree_path"],
            "chunks_path": rec["chunks_path"],
            "score_final": float(final[i]),
            "score_sem_maxsim": float(sem_scores[i]),
            "score_bm25": float(bm25_scores[i]),
            "score_kw": float(kw_scores[i]),
        })

    debug = {
        "sem_scores": sem_scores,
        "bm25_scores": bm25_scores,
        "kw_scores": kw_scores,
        "final_scores": final
    }
    return out, debug

In [ ]:
# Transversal retrieval from provision-aware chunks

def build_faiss_index(embs: np.ndarray) -> faiss.IndexFlatIP:
    embs = embs.astype("float32")
    faiss.normalize_L2(embs)
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)
    return index

def faiss_search(index: faiss.IndexFlatIP, q_emb: np.ndarray, top_n: int):
    q = q_emb.astype("float32").reshape(1, -1)
    faiss.normalize_L2(q)
    scores, idxs = index.search(q, top_n)
    return idxs[0].tolist(), scores[0].tolist()

def get_descendant_raw_ids_from_raw_id(
    raw_node_id: str,
    raw_to_internal: Dict[str, List[str]],
    internal_children_map: Dict[str, List[str]],
    internal_to_raw: Dict[str, str],
    d_max: int = 2,
    N_max: int = 120
) --> List[str]:
    out = []
    seen_internal = set()
    seen_raw = set()

    start_keys = raw_to_internal.get(raw_node_id, [])
    q = deque([(k, 0) for k in start_keys])

    while q and len(out) < N_max:
        cur_key, depth = q.popleft()
        if cur_key in seen_internal:
            continue
        seen_internal.add(cur_key)

        if depth >= d_max:
            continue

        for child_key in internal_children_map.get(cur_key, []):
            child_raw = internal_to_raw.get(child_key, "")

            if IGNORE_NON_NUMERIC_NODE_IDS and not is_structured_regulation_id(child_raw):
                q.append((child_key, depth + 1))
                continue

            if child_raw and child_raw not in seen_raw:
                out.append(child_raw)
                seen_raw.add(child_raw)
                if len(out) >= N_max:
                    break

            q.append((child_key, depth + 1))

    return out

def get_sibling_raw_ids_from_raw_id(
    raw_node_id: str,
    raw_to_internal: Dict[str, List[str]],
    internal_parent_map: Dict[str, str],
    internal_children_map: Dict[str, List[str]],
    internal_to_raw: Dict[str, str]
) -> List[str]:
    sib_raw_ids = []
    seen = set()

    for ikey in raw_to_internal.get(raw_node_id, []):
        pkey = internal_parent_map.get(ikey)
        if pkey is None:
            continue

        for sib_key in internal_children_map.get(pkey, []):
            sib_raw = internal_to_raw.get(sib_key, "")

            if IGNORE_NON_NUMERIC_NODE_IDS and not is_structured_regulation_id(sib_raw):
                continue

            if sib_raw and sib_raw not in seen and sib_raw != raw_node_id:
                sib_raw_ids.append(sib_raw)
                seen.add(sib_raw)

    return sib_raw_ids

def expand_candidates_from_seed_chunks_safe(
    seed_chunk_ids: List[str],
    chunk_meta: Dict[str, Dict[str, Any]],
    node2chunks: Dict[str, List[str]],
    raw_to_internal: Dict[str, List[str]],
    internal_parent_map: Dict[str, str],
    internal_children_map: Dict[str, List[str]],
    internal_to_raw: Dict[str, str],
    d_max: int = 2,
    N_max: int = 120
) -> List[str]:
    cand = set(seed_chunk_ids)

    for cid in seed_chunk_ids:
        raw_node_id = chunk_meta[cid]["node_id"]

        # sibling chunks
        sib_raw_ids = get_sibling_raw_ids_from_raw_id(
            raw_node_id,
            raw_to_internal,
            internal_parent_map,
            internal_children_map,
            internal_to_raw
        )
        for sib_raw in sib_raw_ids:
            for sib_cid in node2chunks.get(sib_raw, []):
                cand.add(sib_cid)

        # descendant chunks
        desc_raw_ids = get_descendant_raw_ids_from_raw_id(
            raw_node_id,
            raw_to_internal,
            internal_children_map,
            internal_to_raw,
            d_max=d_max,
            N_max=N_max
        )
        for desc_raw in desc_raw_ids:
            for desc_cid in node2chunks.get(desc_raw, []):
                cand.add(desc_cid)

        # same node multimodal
        for same_cid in node2chunks.get(raw_node_id, []):
            cand.add(same_cid)

    return list(cand)

def cross_encoder_rerank(
    query: str,
    candidate_chunk_ids: List[str],
    chunk_meta: Dict[str, Dict[str, Any]],
    cross_encoder: CrossEncoder,
    top_k: int = 5,
    batch_size: int = 32
) -> List[Tuple[str, float]]:
    pairs = [(query, chunk_meta[cid]["text"]) for cid in candidate_chunk_ids]
    scores = cross_encoder.predict(pairs, batch_size=batch_size, show_progress_bar=False)
    scores = np.asarray(scores, dtype=float)
    order = np.argsort(-scores)
    return [(candidate_chunk_ids[i], float(scores[i])) for i in order[:top_k]]

def transversal_retrieval(
    query: str,
    doctree: Dict[str, Any],
    chunks: List[Dict[str, Any]],
    seed_top_n: int = 8,
    d_max: int = 2,
    N_max: int = 120,
    final_top_k: int = 5,
    bi_encoder_name: str = "sentence-transformers/all-MiniLM-L6-v2",
    cross_encoder_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2",
):
    for c in chunks:
        if "node_id" not in c or not c["node_id"]:
            if "chunk_id" in c and "::" in c["chunk_id"]:
                c["node_id"] = c["chunk_id"].split("::", 1)[0]
            else:
                raise ValueError(f"Chunk missing node_id: {c}")

        for k in ["chunk_id", "chunk_type", "text"]:
            if k not in c:
                raise ValueError(f"Chunk missing key '{k}': {c}")

    internal_parent_map, internal_children_map, internal_to_raw, raw_to_internal = build_tree_maps_safe(doctree)

    chunk_meta = {c["chunk_id"]: c for c in chunks}
    node2chunks = defaultdict(list)
    for c in chunks:
        node2chunks[c["node_id"]].append(c["chunk_id"])

    bi = SentenceTransformer(bi_encoder_name)
    ce = CrossEncoder(cross_encoder_name)

    chunk_ids = list(chunk_meta.keys())
    texts = [chunk_meta[cid]["text"] for cid in chunk_ids]

    embs = bi.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
    index = build_faiss_index(embs)

    q_emb = bi.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
    seed_idxs, seed_scores = faiss_search(index, q_emb, top_n=min(seed_top_n, len(chunk_ids)))
    seed_chunk_ids = [chunk_ids[i] for i in seed_idxs]

    cand_ids = expand_candidates_from_seed_chunks_safe(
        seed_chunk_ids=seed_chunk_ids,
        chunk_meta=chunk_meta,
        node2chunks=node2chunks,
        raw_to_internal=raw_to_internal,
        internal_parent_map=internal_parent_map,
        internal_children_map=internal_children_map,
        internal_to_raw=internal_to_raw,
        d_max=d_max,
        N_max=N_max
    )

    top = cross_encoder_rerank(
        query=query,
        candidate_chunk_ids=cand_ids,
        chunk_meta=chunk_meta,
        cross_encoder=ce,
        top_k=final_top_k
    )

    retrieval_info = {
        "seed_chunk_ids": seed_chunk_ids,
        "seed_scores": seed_scores,
        "candidate_pool_size": len(cand_ids),
    }

    return top, chunk_meta, retrieval_info

In [ ]:
# Highlight retrieved chunks with respect to the query

def simple_query_terms(query: str) -> List[str]:
    terms = re.findall(r"\b[a-zA-Z][a-zA-Z0-9\-]+\b", query.lower())
    stop = {
        "what", "which", "when", "where", "why", "how", "is", "are", "was", "were",
        "the", "a", "an", "of", "for", "to", "in", "on", "at", "with", "and", "or",
        "by", "from", "that", "this", "these", "those", "be", "as", "it", "its",
        "their", "there", "under", "does", "do", "did", "shall"
    }
    terms = [t for t in terms if t not in stop and len(t) >= 3]

    seen = set()
    out = []
    for t in terms:
        if t not in seen:
            seen.add(t)
            out.append(t)
    return out

def highlight_text_html(text: str, query: str) -> str:
    html = text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    terms = simple_query_terms(query)

    for term in sorted(terms, key=len, reverse=True):
        pattern = re.compile(rf"(?i)\b({re.escape(term)})\b")
        html = pattern.sub(
            r"<mark style='background-color:#fff59d; padding:0 1px;'>\1</mark>",
            html
        )

    return html

In [ ]:
# Self-reflection module with query refinement

def build_evidence_string(
    top_evidence: List[Tuple[str, float]],
    chunk_meta: Dict[str, Dict[str, Any]],
    max_chars_per_chunk: int = 1800
) -> str:
    evidence_blocks = []
    for cid, sc in top_evidence:
        m = chunk_meta[cid]
        txt = m["text"]
        if len(txt) > max_chars_per_chunk:
            txt = txt[:max_chars_per_chunk] + " ...[truncated]"
        evidence_blocks.append(
            f"[{cid}] (type={m['chunk_type']}, node={m['node_id']}, score={sc:.4f})\n{txt}"
        )
    return "\n\n".join(evidence_blocks)

def generate_answer_with_selected_model(
    query: str,
    top_evidence: List[Tuple[str, float]],
    chunk_meta: Dict[str, Dict[str, Any]],
    selected_model_label: str,
    max_chars_per_chunk: int = 1800
) -> str:
    cfg = MODEL_REGISTRY[selected_model_label]
    provider = cfg["provider"]
    model_name = cfg["model_name"]

    evidence_str = build_evidence_string(
        top_evidence=top_evidence,
        chunk_meta=chunk_meta,
        max_chars_per_chunk=max_chars_per_chunk
    )

    system_prompt = (
        "You are a construction regulatory QA assistant. "
        "Answer using only the provided evidence chunks. "
    )

    user_prompt = (
        f"Question:\n{query}\n\n"
        f"Evidence:\n{evidence_str}\n\n"
        f"Provide the answer."
    )

    if provider == "openai":
        if openai_client is None:
            return "[OpenAI API key not set]"
        resp = openai_client.responses.create(
            model=model_name,
            input=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )
        return resp.output_text.strip()

    elif provider == "anthropic":
        if anthropic_client is None:
            return "[Anthropic API key not set]"
        resp = anthropic_client.messages.create(
            model=model_name,
            max_tokens=1024,
            system=system_prompt,
            messages=[{"role": "user", "content": user_prompt}]
        )
        return "".join(
            block.text for block in resp.content if getattr(block, "type", "") == "text"
        ).strip()

    elif provider == "google":
        if google_client is None:
            return "[Google API key not set]"
        full_prompt = f"{system_prompt}\n\n{user_prompt}"
        resp = google_client.models.generate_content(
            model=model_name,
            contents=full_prompt
        )
        return resp.text.strip()

    elif provider == "qwen_local":
        tokenizer, model = load_qwen_once(model_name)
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        inputs = tokenizer(text, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False
            )

        new_tokens = outputs[0][inputs.input_ids.shape[1]:]
        return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    else:
        return f"[Unsupported provider: {provider}]"

def self_reflect_answer(
    query: str,
    initial_answer: str,
    top_evidence: List[Tuple[str, float]],
    chunk_meta: Dict[str, Dict[str, Any]],
    selected_model_label: str
):
    cfg = MODEL_REGISTRY[selected_model_label]
    provider = cfg["provider"]
    model_name = cfg["model_name"]

    evidence_blocks = []
    for cid, sc in top_evidence:
        m = chunk_meta[cid]
        evidence_blocks.append(
            f"[{cid}] (type={m['chunk_type']}, node={m['node_id']}, score={sc:.4f})\n{m['text'][:1500]}"
        )
    evidence_str = "\n\n".join(evidence_blocks)

    system_prompt = (
        "You are a self-reflective evaluator for construction regulatory QA. "
        "Assess whether the draft answer is fully supported by the evidence, "
        "then refine the query if needed to repeat the retrieval process."
    )

    user_prompt = (
        f"Question:\n{query}\n\n"
        f"Draft answer:\n{initial_answer}\n\n"
        f"Evidence:\n{evidence_str}\n\n"
        "Return in this format:\n"
        "Reflection: <brief assessment>\n"
        "Final answer: <improved answer>"
    )

    if provider == "openai":
        if openai_client is None:
            txt = "[OpenAI API key not set]"
        else:
            resp = openai_client.responses.create(
                model=model_name,
                input=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
            )
            txt = resp.output_text.strip()

    elif provider == "anthropic":
        if anthropic_client is None:
            txt = "[Anthropic API key not set]"
        else:
            resp = anthropic_client.messages.create(
                model=model_name,
                max_tokens=1024,
                system=system_prompt,
                messages=[{"role": "user", "content": user_prompt}]
            )
            txt = "".join(
                block.text for block in resp.content if getattr(block, "type", "") == "text"
            ).strip()

    elif provider == "google":
        if google_client is None:
            txt = "[Google API key not set]"
        else:
            resp = google_client.models.generate_content(
                model=model_name,
                contents=f"{system_prompt}\n\n{user_prompt}"
            )
            txt = resp.text.strip()

    elif provider == "qwen_local":
        tokenizer, model = load_qwen_once(model_name)
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        inputs = tokenizer(text, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False
            )

        new_tokens = outputs[0][inputs.input_ids.shape[1]:]
        txt = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    else:
        txt = f"[Unsupported provider: {provider}]"

    reflection = txt
    final_answer = txt

    m1 = re.search(r"Reflection:\s*(.*?)(?:Final answer:|$)", txt, flags=re.S | re.I)
    m2 = re.search(r"Final answer:\s*(.*)", txt, flags=re.S | re.I)

    if m1:
        reflection = m1.group(1).strip()
    if m2:
        final_answer = m2.group(1).strip()

    return {
        "reflection": reflection,
        "final_answer": final_answer
    }

In [ ]:
 # User interface display

 def html_card(title: str, content: str, bg="#fafafa", border="#d9d9d9"):
    return f"""
    <div style="
        border:1px solid {border};
        border-radius:10px;
        padding:12px 14px;
        margin:10px 0;
        background:{bg};
        font-family:Arial,sans-serif;
    ">
      <div style="font-weight:700; font-size:16px; margin-bottom:6px;">{title}</div>
      <div style="white-space:pre-wrap; font-size:14px; line-height:1.45;">{content}</div>
    </div>
    """

def format_division_results(preds):
    lines = []
    for p in preds:
        lines.append(
            f"Rank {p['rank']}: Division {p['division_no']} | {p['document_title']} | "
            f"final={p['score_final']:.4f} | sem={p['score_sem_maxsim']:.4f} | "
            f"bm25={p['score_bm25']:.4f} | kw={p['score_kw']:.4f}"
        )
    return "\n".join(lines)

def summarize_chunks(chunks):
    ct = Counter([c["chunk_type"] for c in chunks])
    return f"Total chunks: {len(chunks)}\nBreakdown: {dict(ct)}"

def make_top_chunks_html(top_evidence, chunk_meta, query):
    blocks = []
    for i, (cid, sc) in enumerate(top_evidence, start=1):
        m = chunk_meta[cid]
        txt = m["text"][:MAX_CHUNK_PREVIEW_CHARS]
        txt_html = highlight_text_html(txt, query)

        block = f"""
        <div style="border:1px solid #ddd; border-radius:8px; padding:10px; margin:8px 0; background:#fffef7;">
            <div style="font-weight:600; margin-bottom:4px;">
                [{i}] {cid} | type={m['chunk_type']} | node={m['node_id']} | score={sc:.4f}
            </div>
            <div style="font-size:14px; line-height:1.5;">{txt_html}</div>
        </div>
        """
        blocks.append(block)
    return "".join(blocks)

In [ ]:
# Main pipeline

def run_pipeline_ui(question: str, selected_model_label: str):
    # Stage 1: Query-guided construction work division identification
    division_preds, _ = rank_divisions_improved(
        question,
        top_n=TOP_K_DIV,
        rrf_k=RRF_K,
        use_weighted=USE_WEIGHTED_FUSION,
        weights=WEIGHTS
    )

    if not division_preds:
        return {"error": "No relevant division identified."}

    selected_div = division_preds[0]
    rec = division_records[selected_div["division_index"]]

    doctree = rec["doctree"]
    chunks = rec["chunks"]
    chunk_summary = summarize_chunks(chunks)

    # Stage 2: Transversal retrieval
    top_evidence, chunk_meta, retrieval_info = transversal_retrieval(
        query=question,
        doctree=doctree,
        chunks=chunks,
        seed_top_n=SEED_TOP_N,
        d_max=D_MAX,
        N_max=N_MAX,
        final_top_k=FINAL_TOP_K
    )

    # Stage 3: Initial answer generation
    initial_answer = generate_answer_with_selected_model(
        query=question,
        top_evidence=top_evidence,
        chunk_meta=chunk_meta,
        selected_model_label=selected_model_label
    )

    # Stage 4: Self-reflection
    reflected = self_reflect_answer(
        query=question,
        initial_answer=initial_answer,
        top_evidence=top_evidence,
        chunk_meta=chunk_meta,
        selected_model_label=selected_model_label
    )

    return {
        "question": question,
        "selected_model_label": selected_model_label,
        "selected_model_api_name": MODEL_REGISTRY[selected_model_label]["model_name"],
        "division_preds": division_preds,
        "selected_division": selected_div,
        "document_title": doctree.get("document_title", "Untitled"),
        "chunk_summary": chunk_summary,
        "retrieval_info": retrieval_info,
        "top_evidence": top_evidence,
        "chunk_meta": chunk_meta,
        "initial_answer": initial_answer,
        "reflection": reflected["reflection"],
        "final_answer": reflected["final_answer"],
    }


 # User interface display

default_question = "What are the definitions of bead and butt weld?"

title_html = widgets.HTML(
    """
    <h2 style='font-family:Arial; margin-bottom:8px;'>
        Interactive user interface for the proposed pipeline
    </h2>
    """
)

question_box = widgets.Textarea(
    value=default_question,
    placeholder="Enter one question here...",
    description="Question:",
    layout=widgets.Layout(width="100%", height="100px")
)

model_selector = widgets.Dropdown(
    options=list(MODEL_REGISTRY.keys()),
    value=DEFAULT_MODEL_LABEL,
    description="Model:",
    layout=widgets.Layout(width="420px")
)

run_button = widgets.Button(
    description="Run pipeline",
    button_style="primary",
    layout=widgets.Layout(width="180px")
)

status_out = widgets.Output()
stage_out = widgets.Output()

def on_run_clicked(b):
    with status_out:
        clear_output()
        print("Running pipeline...")

    with stage_out:
        clear_output()

        q = question_box.value.strip()
        selected_model_label = model_selector.value

        if not q:
            display(HTML(html_card(
                "Error",
                "Please enter a question.",
                bg="#fff4f4",
                border="#e39b9b"
            )))
            return

        try:
            t0 = time.perf_counter()
            result = run_pipeline_ui(q, selected_model_label=selected_model_label)
            t1 = time.perf_counter()

            if "error" in result:
                display(HTML(html_card(
                    "Error",
                    result["error"],
                    bg="#fff4f4",
                    border="#e39b9b"
                )))
                return

            display(HTML(html_card(
                "Model selection",
                f"Selected model family: {result['selected_model_label']}\n"
                f"API / checkpoint name: {result['selected_model_api_name']}",
                bg="#f3f8ff",
                border="#8fb7ff"
            )))

            display(HTML(html_card(
                "Step 1 — Division identification (Query-guided construction work division identification)",
                f"Top ranked divisions:\n{format_division_results(result['division_preds'])}\n\n"
                f"Selected division:\nDivision {result['selected_division']['division_no']} | "
                f"{result['selected_division']['document_title']}",
                bg="#f7fbff",
                border="#9ec5fe"
            )))

            display(HTML(html_card(
                "Step 2 — Hierarchical provision-aware chunking (Division organization-aligned chunking)",
                f"Division title: {result['document_title']}\n\n{result['chunk_summary']}",
                bg="#f8fff7",
                border="#94d39b"
            )))

            display(HTML(html_card(
                "Step 3 — Transversal retrieval",
                f"Seed chunk count: {len(result['retrieval_info']['seed_chunk_ids'])}\n"
                f"Expanded candidate pool size: {result['retrieval_info']['candidate_pool_size']}",
                bg="#fffdf5",
                border="#e6cc7a"
            )))

            display(HTML(
                "<div style='font-family:Arial; font-size:16px; font-weight:700; margin:10px 0;'>"
                "Top evidence chunks (query-related terms highlighted)"
                "</div>"
            ))
            display(HTML(make_top_chunks_html(result["top_evidence"], result["chunk_meta"], q)))

            display(HTML(html_card(
                "Step 4 — Initial answer generation",
                result["initial_answer"],
                bg="#f7f7f7",
                border="#cfcfcf"
            )))

            display(HTML(html_card(
                "Step 5 — Self-reflective evaluation",
                result["reflection"],
                bg="#f9f4ff",
                border="#bca3e3"
            )))

            display(HTML(html_card(
                "Step 6 — Final refined answer",
                result["final_answer"],
                bg="#eef9f0",
                border="#8fcd9a"
            )))

            display(HTML(html_card(
                "Runtime",
                f"Total elapsed time: {t1 - t0:.2f} s",
                bg="#fcfcfc",
                border="#dddddd"
            )))

        except Exception as e:
            display(HTML(html_card(
                "Pipeline error",
                str(e),
                bg="#fff4f4",
                border="#e39b9b"
            )))

    with status_out:
        clear_output()
        print("Done.")

run_button.on_click(on_run_clicked)

ui = widgets.VBox([
    title_html,
    model_selector,
    question_box,
    run_button,
    status_out,
    stage_out
])

display(ui)